**Lab type:** debug  
**Course:** DS104 — Statistics for Data Science  
**Lesson:** Hypothesis Testing in Practice  
**Task:** The hypothesis testing pipeline below contains 3 bugs. Each runs without errors but produces wrong conclusions — through wrong test selection, multiple comparison inflation, or conflating statistical with practical significance. Find each bug, explain it, and fix it.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(21)

# Dataset: A/B test comparing two onboarding flows
# Control: old flow, n=40 users
# Treatment: new flow, n=38 users
# Outcome: 7-day activity score (0-100), both groups are non-normal (skewed)

# Both drawn from exponential to ensure non-normality
control = np.random.exponential(scale=28, size=40).clip(0, 100)
treatment = np.random.exponential(scale=32, size=38).clip(0, 100)

print('Control group:')
print(f'  n={len(control)}, mean={control.mean():.2f}, median={np.median(control):.2f}')
print('Treatment group:')
print(f'  n={len(treatment)}, mean={treatment.mean():.2f}, median={np.median(treatment):.2f}')

## Bug 1: t-test applied without checking normality

The code below runs a two-sample t-test without first verifying whether the normality assumption holds.

In [ ]:
# --- BUGGY CODE ---
t_stat, p_value = stats.ttest_ind(control, treatment)

print(f't-statistic: {t_stat:.3f}')
print(f'p-value: {p_value:.4f}')
if p_value < 0.05:
    print('Result: Statistically significant — new onboarding flow improves activity.')
else:
    print('Result: Not significant.')

**Explanation:** Write your answer here — why is a t-test inappropriate here? What assumption does it require that is likely violated, and how would you check it?

*(Write your answer here.)*

**Fix the bug: check assumptions first, then select the correct test.**

In [ ]:
# Step 1: Check normality with Shapiro-Wilk
_, p_norm_control = stats.shapiro(control)
_, p_norm_treatment = stats.shapiro(treatment)

print(f'Shapiro-Wilk: control p={p_norm_control:.4f}, treatment p={p_norm_treatment:.4f}')
print(f'Both normal: {p_norm_control > 0.05 and p_norm_treatment > 0.05}')

# Step 2: Since data is non-normal and n < 30, use Mann-Whitney U
u_stat, p_mw = stats.mannwhitneyu(control, treatment, alternative='two-sided')

# Effect size: rank-biserial correlation
n1, n2 = len(control), len(treatment)
r_rb = 1 - (2 * u_stat) / (n1 * n2)

print(f'\nMann-Whitney U: U={u_stat:.0f}, p={p_mw:.4f}')
print(f'Effect size (rank-biserial r): {r_rb:.3f}')

interpretation = (
    'Statistically significant' if p_mw < 0.05 else 'Not significant'
)
print(f'Result: {interpretation}. Effect size: {abs(r_rb):.2f} (small < 0.1, medium 0.1–0.3, large > 0.3).')

## Bug 2: Multiple comparisons without correction

An analyst screens 10 features for association with churn. Each test is run at α = 0.05 with no correction.

In [ ]:
# Simulate 10 features where the null is TRUE for all (no real effect)
np.random.seed(88)
n_churned, n_retained = 500, 1500
n_tests = 10

results = []
for i in range(n_tests):
    # Both groups drawn from the same distribution — no real difference
    churned_vals = np.random.normal(50, 15, n_churned)
    retained_vals = np.random.normal(50, 15, n_retained)
    _, p = stats.ttest_ind(churned_vals, retained_vals)
    results.append({'feature': f'feature_{i+1}', 'p_value': p})

results_df = pd.DataFrame(results)

# --- BUGGY CODE ---
# No multiple comparison correction
significant = results_df[results_df.p_value < 0.05]

print('--- BUGGY: No Bonferroni correction ---')
print(f'Tests run: {n_tests}, alpha per test: 0.05')
print(f'"Significant" features (all are false positives!): {len(significant)}')
print(significant[['feature', 'p_value']].to_string(index=False))
print(f'Expected false positives under null: {n_tests * 0.05:.1f}')

**Explanation:** Write your answer here — why does running 10 tests at α=0.05 inflate the false-positive rate? What correction should be applied?

*(Write your answer here.)*

**Fix the bug:**

In [ ]:
# Bonferroni correction: divide alpha by number of tests
alpha_corrected = 0.05 / n_tests

significant_corrected = results_df[results_df.p_value < alpha_corrected]

print(f'Bonferroni-corrected alpha: 0.05 / {n_tests} = {alpha_corrected:.4f}')
print(f'Significant features after correction: {len(significant_corrected)}')
print()
print(results_df.sort_values('p_value').to_string(index=False))

## Bug 3: Practical significance ignored — p-value only

A large-scale experiment reports a statistically significant improvement. The developer declares success without checking effect size.

In [ ]:
# Large A/B test: 15,000 users per group
np.random.seed(55)
n_large = 15000

# Conversion rates: control=24.0%, treatment=24.8% — tiny real difference
control_conversions = np.random.binomial(1, 0.240, n_large)
treatment_conversions = np.random.binomial(1, 0.248, n_large)

# --- BUGGY CODE ---
t_stat, p_value = stats.ttest_ind(control_conversions, treatment_conversions)

print(f'Control conversion rate: {control_conversions.mean():.4f} ({control_conversions.mean()*100:.2f}%)')
print(f'Treatment conversion rate: {treatment_conversions.mean():.4f} ({treatment_conversions.mean()*100:.2f}%)')
print(f'p-value: {p_value:.4f}')

# BUG: success declared based on p-value alone
if p_value < 0.05:
    print('SUCCESS: New landing page significantly improves conversion. Ship it!')

**Explanation:** Write your answer here — the result is statistically significant with p < 0.05. Why is declaring success premature? What other information do you need before a shipping decision?

*(Write your answer here.)*

**Fix: report effect size and make the business case explicit.**

In [ ]:
# Compute Cohen's d for effect size
pooled_std = np.sqrt(
    (control_conversions.std()**2 + treatment_conversions.std()**2) / 2
)
cohens_d = (treatment_conversions.mean() - control_conversions.mean()) / pooled_std

absolute_lift = treatment_conversions.mean() - control_conversions.mean()
relative_lift = absolute_lift / control_conversions.mean()

print(f'p-value: {p_value:.4f} (statistically significant at α=0.05)')
print(f"Cohen's d: {cohens_d:.4f} (tiny — threshold for 'small' is d=0.2)")
print(f'Absolute lift: {absolute_lift*100:.2f} percentage points')
print(f'Relative lift: {relative_lift*100:.1f}%')
print()
print('Business framing: Is a 0.8pp lift worth the development cost and rollout risk?')
print('That is a business judgment — the p-value alone cannot make it.')

## Summary

> **Final question:** In one sentence each, state the three lessons from this lab.

1. 
2. 
3. 